# Interactive exploration

An **analysis/teaching layer** on top of the library — not the reproducible
experiment path. Drag the sliders to build intuition: change the population mix,
the information model, and the agents' ecological knowledge, and watch the
resource trajectory and summary metrics update.

For reproducible results use the CLI (`emergent-coop run --config ...`) and the
scripts in `scripts/`; the findings are written up in
[`docs/findings-summary.md`](../docs/findings-summary.md).

**Setup:** `pip install -e ".[notebook]"` then run all cells.

> ⚠️ **Kernel:** if you get `ModuleNotFoundError: No module named 'emergent_cooperation'`,
> the notebook is running under the wrong Python. Use **Kernel → Change Kernel** and
> pick **"emergent-coop (.venv)"** (the project's `.venv`), *not* the default
> "Python 3 (ipykernel)". To register the kernel once:
> `python -m ipykernel install --user --name emergent-coop --display-name "emergent-coop (.venv)"`
> (run it from the activated `.venv`).

In [ ]:
import matplotlib.pyplot as plt

from emergent_cooperation.core.config import AgentSpec, ResourceConfig, SimulationConfig
from emergent_cooperation.core.simulation import run_simulation
from emergent_cooperation.metrics.metrics import compute_metrics

GROUP_SIZE = 8


def explore(cooperator_type="cooperative", n_selfish=0, information_model="global",
            knowledge_bias=1.0, rounds=60, seed=1):
    """Run one scenario and plot the resource trajectory + summary metrics."""
    n_coop = GROUP_SIZE - n_selfish
    params = {"regeneration_rate": 0.4, "capacity": 100.0}
    if cooperator_type in ("cooperative", "conditional_cooperator"):
        params["knowledge_bias"] = knowledge_bias
    if cooperator_type == "sanctioning":
        params["monitoring_cost"] = 0.2

    agents = []
    if n_coop > 0:
        agents.append(AgentSpec(cooperator_type, n_coop, params))
    if n_selfish > 0:
        agents.append(AgentSpec("selfish", n_selfish, {"greed": 1.0}))

    cfg = SimulationConfig(
        name="explore", rounds=rounds, information_model=information_model,
        resource=ResourceConfig(initial_level=50.0, capacity=100.0,
                                regeneration_rate=0.4, collapse_threshold=1.0),
        agents=tuple(agents),
    )
    result = run_simulation(cfg, seed=seed)
    m = compute_metrics(result, capacity=100.0, regeneration_rate=0.4, collapse_threshold=1.0)

    x = [r.round_index for r in result.rounds]
    stock = [r.resource_after_harvest for r in result.rounds]
    harvest = [r.total_harvested for r in result.rounds]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(x, stock); ax1.axhline(50, color="grey", ls=":", lw=1)
    ax1.set_ylim(-2, 100); ax1.set_xlabel("round"); ax1.set_ylabel("standing stock")
    ax1.set_title("resource over time")
    ax2.plot(x, harvest, color="tab:green"); ax2.axhline(10, color="grey", ls=":", lw=1)
    ax2.set_xlabel("round"); ax2.set_ylabel("total harvest"); ax2.set_title("harvest over time")
    fig.suptitle(
        f"{n_coop} {cooperator_type} + {n_selfish} selfish  [{information_model}]   "
        f"sustainability={m['sustainability_ratio']:.2f}  "
        f"collapsed={m['collapsed']}  gini={m['payoff_gini']:.2f}"
    )
    fig.tight_layout(); plt.show()
    return m

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

interact(
    explore,
    cooperator_type=widgets.Dropdown(
        options=["cooperative", "conditional_cooperator", "sanctioning"], value="cooperative"),
    n_selfish=widgets.IntSlider(min=0, max=8, value=0, description="# selfish"),
    information_model=widgets.Dropdown(options=["global", "private"], value="global"),
    knowledge_bias=widgets.FloatSlider(min=0.6, max=1.5, step=0.1, value=1.0,
                                       description="knowledge"),
    rounds=widgets.IntSlider(min=20, max=100, step=10, value=60),
    seed=widgets.IntSlider(min=1, max=10, value=1),
);

### Things to try

- **`cooperative`, `private`, knowledge = 1.3** — overconfident blind cooperators
  collapse the pool (E1 / hypothesis H6).
- **`conditional_cooperator`, # selfish = 1** — one free-rider triggers a
  retaliation ratchet to collapse (E2).
- **`sanctioning`, # selfish = 4** — enforcement holds the resource steady and keeps
  Gini low, even against free-riders (E3).
- Compare **`global` vs `private`** for the cooperative type at different knowledge
  values — information substitutes for knowledge.